## Лабораторная работа №3
### Подготовка выборки, кросс-валидация и подбор гиперпараметров для метода k‑ближайших соседей

**Цель работы** – изучить:

- разделение данных на обучающую и тестовую выборки;
- реализацию метода k‑ближайших соседей (k‑NN) для классификации;
- оценку качества модели;
- подбор гиперпараметра *K* с помощью GridSearchCV и RandomizedSearchCV с разными стратегиями кросс‑валидации.

В качестве датасета используется **Titanic** (после предобработки). Задача – бинарная классификация (выжил / не выжил).

## 1. Краткое описание датасета Titanic

Файл `titanic.csv` содержит данные о пассажирах.  
Целевая переменная – `Survived` (1 – выжил, 0 – погиб).  
После очистки будем использовать следующие признаки:

- `Pclass` – класс билета (числовой)
- `Age` – возраст (заполнен медианой)
- `SibSp` – количество братьев/сестёр/супругов
- `Parch` – количество родителей/детей
- `Fare` – стоимость билета
- `Sex` – пол (закодирован 0/1)
- `Embarked` – порт посадки (One‑Hot Encoding)

Пропуски заполнены, категориальные признаки закодированы, все признаки масштабированы.

In [1]:
#r "nuget: Microsoft.Data.Analysis, 0.22.1"
#r "nuget: Microsoft.ML, 3.0.1"
#r "nuget: Plotly.NET.Interactive, 5.0.0"
#r "nuget: Plotly.NET.CSharp, 0.12.0"
#r "nuget: MathNet.Numerics, 5.0.0"

The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages MathNet.Numerics, 5.0.0 Microsoft.Data.Analysis, 0.22.1 Microsoft.ML, 3.0.1 Plotly.NET.CSharp, 0.12.0 Plotly.NET.Interactive, 5.0.0

Loading extensions from `C:\Users\0\.nuget\packages\plotly.net.interactive\5.0.0\lib\netstandard2.1\Plotly.NET.Interactive.dll`

Loading extensions from `C:\Users\0\.nuget\packages\microsoft.data.analysis\0.22.1\interactive-extensions\dotnet\Microsoft.Data.Analysis.Interactive.dll`

In [2]:
using Plotly.NET;
using PlotlyCS = Plotly.NET.CSharp.Chart;
using Microsoft.Data.Analysis;
using Microsoft.ML;
using MathNet.Numerics.Statistics;
using Microsoft.DotNet.Interactive.Formatting;
using System.Linq;
using System.IO;
using System.Net;
using System.Globalization;

In [3]:
var dataDir = Path.Combine(Directory.GetCurrentDirectory(), "Data");
Directory.CreateDirectory(dataDir);
var dataPath = Path.Combine(dataDir, "titanic.csv");

if (!File.Exists(dataPath))
{
    var url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv";
    Console.WriteLine("Скачиваю titanic.csv...");
    new WebClient().DownloadFile(url, dataPath);
    Console.WriteLine("Готово.");
}

// Явное задание имён и типов столбцов
string[] colNames = { "PassengerId", "Survived", "Pclass", "Name", "Sex", "Age", "SibSp", "Parch", "Ticket", "Fare", "Cabin", "Embarked" };
Type[] colTypes = { typeof(float), typeof(float), typeof(float), typeof(string), typeof(string), typeof(float), typeof(float), typeof(float), typeof(string), typeof(float), typeof(string), typeof(string) };

var df = DataFrame.LoadCsv(dataPath,
    separator: ',',
    header: true,
    columnNames: colNames,
    dataTypes: colTypes,
    cultureInfo: CultureInfo.InvariantCulture);

Console.WriteLine("Столбцы: " + string.Join(", ", df.Columns.Select(c => c.Name)));
df.Head(3).Display();

Столбцы: PassengerId, Survived, Pclass, Name, Sex, Age, SibSp, Parch, Ticket, Fare, Cabin, Embarked


index,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22,1,0,A/5 21171,7.25,,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Thayer)",female,38,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26,0,0,STON/O2. 3101282,7.925,,S



(9,5): warning SYSLIB0014: "WebClient.WebClient()" является устаревшим: 'WebRequest, HttpWebRequest, ServicePoint, and WebClient are obsolete. Use HttpClient instead.' (https://aka.ms/dotnet-warnings/SYSLIB0014)



## 2. Предварительная обработка данных

Выполним:

- удаление бесполезных столбцов (`PassengerId`, `Name`, `Ticket`, `Cabin`);
- заполнение пропусков в `Age` (медиана) и `Embarked` (мода);
- кодирование `Sex` (Label Encoding) и `Embarked` (One‑Hot Encoding);
- масштабирование числовых признаков (StandardScaler).

In [4]:
df.Columns.Remove("PassengerId");
df.Columns.Remove("Name");
df.Columns.Remove("Ticket");
df.Columns.Remove("Cabin");
Console.WriteLine("Оставшиеся столбцы: " + string.Join(", ", df.Columns.Select(c => c.Name)));

Оставшиеся столбцы: Survived, Pclass, Sex, Age, SibSp, Parch, Fare, Embarked


In [5]:
var ageCol = df["Age"];

// Собираем все реальные значения (не null)
var validAges = new List<float>();
for (long i = 0; i < df.Rows.Count; i++)
{
    if (ageCol[i] is float f)
        validAges.Add(f);
}

// Вычисляем медиану
validAges.Sort();
float medianAge = validAges[validAges.Count / 2];
Console.WriteLine($"Медиана Age: {medianAge}");

// Заменяем null на медиану
for (long i = 0; i < df.Rows.Count; i++)
{
    if (ageCol[i] is null)
        ageCol[i] = medianAge;
}

// Проверка: считаем пропуски вручную
long ageNulls = 0;
for (long i = 0; i < df.Rows.Count; i++)
{
    if (ageCol[i] is null)
        ageNulls++;
}
Console.WriteLine($"Пропусков в Age после заполнения: {ageNulls}");

Медиана Age: 28
Пропусков в Age после заполнения: 0


In [6]:
var embarkedCol = df["Embarked"];
var modeEmbarked = embarkedCol.Cast<string>()
    .Where(s => s != null)
    .GroupBy(s => s)
    .OrderByDescending(g => g.Count())
    .First().Key;
Console.WriteLine($"Мода Embarked: {modeEmbarked}");

for (long i = 0; i < df.Rows.Count; i++)
{
    if (embarkedCol[i] is null)
        embarkedCol[i] = modeEmbarked;
}

Мода Embarked: S


In [7]:
var sexCol = df["Sex"];
var sexValues = new int[df.Rows.Count];
for (long i = 0; i < df.Rows.Count; i++)
{
    sexValues[i] = (string)sexCol[i] == "male" ? 1 : 0;
}
df.Columns.Remove("Sex");
df.Columns.Add(new PrimitiveDataFrameColumn<int>("Sex", sexValues));
Console.WriteLine("Sex закодирован (0 – female, 1 – male)");

Sex закодирован (0 – female, 1 – male)


In [8]:
var embarkedValues = df["Embarked"].Cast<string>().ToArray();
var categories = new[] { "S", "C", "Q" };

foreach (var cat in categories)
{
    var values = embarkedValues.Select(v => v == cat ? 1 : 0).ToArray();
    df.Columns.Add(new PrimitiveDataFrameColumn<int>($"Embarked_{cat}", values));
}
df.Columns.Remove("Embarked");
Console.WriteLine("Добавлены столбцы: " + string.Join(", ", categories.Select(c => "Embarked_" + c)));

Добавлены столбцы: Embarked_S, Embarked_C, Embarked_Q


In [9]:
// Преобразуем int-столбцы во float (кроме Survived, который мы не масштабируем)
string[] intColumns = { "Pclass", "SibSp", "Parch", "Sex", "Embarked_S", "Embarked_C", "Embarked_Q" };

foreach (var colName in intColumns)
{
    var floatValues = new float[df.Rows.Count];
    for (long i = 0; i < df.Rows.Count; i++)
    {
        var val = df[colName][i];
        if (val is int iv)
            floatValues[i] = iv;
        else if (val is float fv)
            floatValues[i] = fv;
        else
            floatValues[i] = 0f;
    }
    
    df.Columns.Remove(colName);
    df.Columns.Add(new PrimitiveDataFrameColumn<float>(colName, floatValues));
}

// Функция масштабирования
void Standardize(DataFrame df, string colName)
{
    var col = df[colName];
    var vals = new List<float>();
    for (long i = 0; i < df.Rows.Count; i++)
    {
        if (col[i] is float f)
            vals.Add(f);
    }
    double mean = vals.Average();
    double std = Math.Sqrt(vals.Average(v => (v - mean) * (v - mean)));
    for (long i = 0; i < df.Rows.Count; i++)
    {
        if (col[i] is float f)
        {
            col[i] = (float)((f - mean) / std);
        }
    }
}

// Масштабируем все числовые признаки
Standardize(df, "Age");
Standardize(df, "Fare");
Standardize(df, "Pclass");
Standardize(df, "SibSp");
Standardize(df, "Parch");
Standardize(df, "Sex");
Standardize(df, "Embarked_S");
Standardize(df, "Embarked_C");
Standardize(df, "Embarked_Q");

Console.WriteLine("Все признаки масштабированы (StandardScaler).");

Все признаки масштабированы (StandardScaler).


In [10]:
// Формируем список столбцов-признаков (все кроме Survived)
var featureCols = df.Columns.Where(c => c.Name != "Survived").Select(c => c.Name).ToArray();
Console.WriteLine("Признаки: " + string.Join(", ", featureCols));

int nRows = (int)df.Rows.Count;
int nFeatures = featureCols.Length;
double[][] X = new double[nRows][];
double[] y = new double[nRows];

for (int i = 0; i < nRows; i++)
{
    X[i] = new double[nFeatures];
    for (int j = 0; j < nFeatures; j++)
    {
        var val = df[featureCols[j]][i];
        if (val is float f) X[i][j] = f;
        else if (val is int iv) X[i][j] = iv;
        else X[i][j] = 0.0;
    }
    y[i] = (float)df["Survived"][i];
}

Console.WriteLine($"Матрица признаков: {X.Length} строк, {nFeatures} столбцов");

Признаки: Age, Fare, Pclass, SibSp, Parch, Sex, Embarked_S, Embarked_C, Embarked_Q
Матрица признаков: 891 строк, 9 столбцов


In [11]:
var survivedCounts = df["Survived"].Cast<float>()
    .GroupBy(v => v)
    .ToDictionary(g => g.Key == 1.0 ? "Выжил" : "Погиб", g => g.Count());

var pieChart = PlotlyCS.Pie<int, string, string>(
    values: survivedCounts.Values.ToArray(),
    Labels: survivedCounts.Keys.ToArray()
).WithTitle("Соотношение классов в полном датасете");

display(pieChart);

<!-- Plotly chart will be drawn inside this DIV -->

## 3. Разделение на обучающую и тестовую выборки (train_test_split)

In [12]:
(int[][] train, int[][] test) TrainTestSplit(int[] indices, double testSize = 0.2, int seed = 42)
{
    var rnd = new Random(seed);
    var shuffled = indices.OrderBy(i => rnd.Next()).ToArray();
    int testCount = (int)(shuffled.Length * testSize);
    var testIndices = shuffled.Take(testCount).ToArray();
    var trainIndices = shuffled.Skip(testCount).ToArray();
    return (new[] { trainIndices }, new[] { testIndices });
}

int[] allIndices = Enumerable.Range(0, X.Length).ToArray();
var split = TrainTestSplit(allIndices, 0.2, 42);
var trainIdx = split.train[0];
var testIdx = split.test[0];

double[][] X_train = trainIdx.Select(i => X[i]).ToArray();
double[] y_train = trainIdx.Select(i => y[i]).ToArray();
double[][] X_test = testIdx.Select(i => X[i]).ToArray();
double[] y_test = testIdx.Select(i => y[i]).ToArray();

Console.WriteLine($"Обучающая выборка: {X_train.Length} объектов");
Console.WriteLine($"Тестовая выборка: {X_test.Length} объектов");

// Визуализация распределения классов в train и test (Pie charts)
var trainCounts = y_train.GroupBy(v => v).ToDictionary(g => g.Key == 1.0 ? "Выжил" : "Погиб", g => g.Count());
display(PlotlyCS.Pie<int, string, string>(values: trainCounts.Values.ToArray(), Labels: trainCounts.Keys.ToArray())
    .WithTitle("Обучающая выборка"));

var testCounts = y_test.GroupBy(v => v).ToDictionary(g => g.Key == 1.0 ? "Выжил" : "Погиб", g => g.Count());
display(PlotlyCS.Pie<int, string, string>(values: testCounts.Values.ToArray(), Labels: testCounts.Keys.ToArray())
    .WithTitle("Тестовая выборка"));

Обучающая выборка: 713 объектов
Тестовая выборка: 178 объектов


<!-- Plotly chart will be drawn inside this DIV -->

<!-- Plotly chart will be drawn inside this DIV -->

## 4. Классификатор k‑ближайших соседей

In [13]:
public class KNNClassifier
{
    private double[][] _X;
    private double[] _y;
    private int _k;
    
    public KNNClassifier(int k)
    {
        _k = k;
    }
    
    public void Fit(double[][] X, double[] y)
    {
        _X = X;
        _y = y;
    }
    
    public double[] Predict(double[][] X)
    {
        var preds = new double[X.Length];
        for (int i = 0; i < X.Length; i++)
            preds[i] = PredictSingle(X[i]);
        return preds;
    }
    
    private double PredictSingle(double[] x)
    {
        int n = _X.Length;
        var distances = new (double dist, double label)[n];
        for (int i = 0; i < n; i++)
        {
            double d = 0.0;
            for (int j = 0; j < x.Length; j++)
            {
                double diff = _X[i][j] - x[j];
                d += diff * diff;
            }
            distances[i] = (Math.Sqrt(d), _y[i]);
        }
        
        var nearest = distances.OrderBy(t => t.dist).Take(_k);
        double sum = nearest.Sum(t => t.label);
        return sum > _k / 2.0 ? 1.0 : 0.0;
    }
}

In [14]:
double Accuracy(double[] yTrue, double[] yPred)
{
    int correct = 0;
    for (int i = 0; i < yTrue.Length; i++)
        if ((int)yTrue[i] == (int)yPred[i]) correct++;
    return (double)correct / yTrue.Length;
}

var knnBase = new KNNClassifier(5);
knnBase.Fit(X_train, y_train);
var predBase = knnBase.Predict(X_test);

double baseAcc = Accuracy(y_test, predBase);
Console.WriteLine($"Точность базовой модели (K=5): {baseAcc:F4}");

// Матрица ошибок
int tp = 0, tn = 0, fp = 0, fn = 0;
for (int i = 0; i < y_test.Length; i++)
{
    int actual = (int)y_test[i];
    int pred = (int)predBase[i];
    if (actual == 1 && pred == 1) tp++;
    else if (actual == 1 && pred == 0) fn++;
    else if (actual == 0 && pred == 1) fp++;
    else if (actual == 0 && pred == 0) tn++;
}
Console.WriteLine($"Confusion Matrix:\nTP={tp}, FP={fp}\nFN={fn}, TN={tn}");

// Столбчатая диаграмма матрицы ошибок
var metrics = new[] { "TP", "FP", "FN", "TN" };
var metricValues = new[] { (double)tp, fp, fn, tn };
var barChart = PlotlyCS.Column<string, double, string>(metrics, metricValues)
    .WithTitle("Матрица ошибок (K=5)");
display(barChart);

Точность базовой модели (K=5): 0,7809
Confusion Matrix:
TP=50, FP=15
FN=24, TN=89


<!-- Plotly chart will be drawn inside this DIV -->

## 5. Кросс-валидация и подбор гиперпараметра K

In [15]:
IEnumerable<(int[] train, int[] val)> KFold(int[] indices, int folds = 5, int seed = 42)
{
    var rnd = new Random(seed);
    var shuffled = indices.OrderBy(i => rnd.Next()).ToArray();
    int foldSize = (int)Math.Ceiling((double)shuffled.Length / folds);
    for (int f = 0; f < folds; f++)
    {
        var val = shuffled.Skip(f * foldSize).Take(foldSize).ToArray();
        var train = shuffled.Except(val).ToArray();
        yield return (train, val);
    }
}

double EvaluateK(int k, double[][] allX, double[] allY, int[] indices, int folds = 5)
{
    double sumAcc = 0.0;
    int count = 0;
    foreach (var (trainIdx, valIdx) in KFold(indices, folds))
    {
        var X_tr = trainIdx.Select(i => allX[i]).ToArray();
        var y_tr = trainIdx.Select(i => allY[i]).ToArray();
        var X_val = valIdx.Select(i => allX[i]).ToArray();
        var y_val = valIdx.Select(i => allY[i]).ToArray();
        
        var knn = new KNNClassifier(k);
        knn.Fit(X_tr, y_tr);
        var preds = knn.Predict(X_val);
        sumAcc += Accuracy(y_val, preds);
        count++;
    }
    return sumAcc / count;
}

In [16]:
(int bestK, double bestScore, Dictionary<int, double> allScores) GridSearchDetailed(int[] kValues, double[][] X, double[] y, int folds = 5)
{
    int[] indices = Enumerable.Range(0, X.Length).ToArray();
    int bestK = kValues[0];
    double bestScore = 0.0;
    var scores = new Dictionary<int, double>();
    foreach (int k in kValues)
    {
        double score = EvaluateK(k, X, y, indices, folds);
        scores[k] = score;
        Console.WriteLine($"K={k}: средняя точность CV = {score:F4}");
        if (score > bestScore)
        {
            bestScore = score;
            bestK = k;
        }
    }
    return (bestK, bestScore, scores);
}

int[] kRange = Enumerable.Range(1, 15).Select(i => i * 2 - 1).ToArray(); // 1,3,5,...,29
Console.WriteLine("GridSearchCV (5-фолдь):");
var (bestK_Grid, bestScore_Grid, gridScores) = GridSearchDetailed(kRange, X_train, y_train, folds: 5);
Console.WriteLine($"\nЛучший K (GridSearch): {bestK_Grid} с точностью {bestScore_Grid:F4}");

// График точности от K
var kAxis = gridScores.Keys.Select(k => (double)k).ToArray();
var accAxis = gridScores.Values.ToArray();
var lineChart = PlotlyCS.Line<double, double, string>(kAxis, accAxis)
    .WithTitle("Точность кросс-валидации от K (GridSearch)")
    .WithXAxisStyle(Title.init("K"))
    .WithYAxisStyle(Title.init("Accuracy"));
display(lineChart);

GridSearchCV (5-фолдь):
K=1: средняя точность CV = 0,7379
K=3: средняя точность CV = 0,7897
K=5: средняя точность CV = 0,8093
K=7: средняя точность CV = 0,8178
K=9: средняя точность CV = 0,8234
K=11: средняя точность CV = 0,8332
K=13: средняя точность CV = 0,8276
K=15: средняя точность CV = 0,8164
K=17: средняя точность CV = 0,8192
K=19: средняя точность CV = 0,8262
K=21: средняя точность CV = 0,8178
K=23: средняя точность CV = 0,8178
K=25: средняя точность CV = 0,8094
K=27: средняя точность CV = 0,8122
K=29: средняя точность CV = 0,7911

Лучший K (GridSearch): 11 с точностью 0,8332


<!-- Plotly chart will be drawn inside this DIV -->

In [17]:
(int bestK, double bestScore, List<(int K, double Score)> allResults) RandomizedSearchDetailed(
    int[] kPool, int nIter, double[][] X, double[] y, int folds = 5, int seed = 42)
{
    var rnd = new Random(seed);
    int[] indices = Enumerable.Range(0, X.Length).ToArray();
    int bestK = kPool[0];
    double bestScore = 0.0;
    var results = new List<(int K, double Score)>();
    for (int iter = 0; iter < nIter; iter++)
    {
        int k = kPool[rnd.Next(kPool.Length)];
        double score = EvaluateK(k, X, y, indices, folds);
        results.Add((k, score));
        Console.WriteLine($"Итерация {iter+1}: K={k}, точность={score:F4}");
        if (score > bestScore)
        {
            bestScore = score;
            bestK = k;
        }
    }
    return (bestK, bestScore, results);
}

Console.WriteLine("\nRandomizedSearchCV (5-фолдь, 10 итераций):");
var (bestK_Rand, bestScore_Rand, randResults) = RandomizedSearchDetailed(kRange, 10, X_train, y_train, folds: 5);
Console.WriteLine($"\nЛучший K (Randomized): {bestK_Rand} с точностью {bestScore_Rand:F4}");

// Точечный график результатов RandomizedSearch
var scatterRand = PlotlyCS.Scatter<int, double, string>(
    randResults.Select(r => r.K).ToArray(),
    randResults.Select(r => r.Score).ToArray(),
    mode: StyleParam.Mode.Markers
).WithTitle("Результаты RandomizedSearchCV")
 .WithXAxisStyle(Title.init("K"))
 .WithYAxisStyle(Title.init("Accuracy"));
display(scatterRand);


RandomizedSearchCV (5-фолдь, 10 итераций):
Итерация 1: K=21, точность=0,8178
Итерация 2: K=5, точность=0,8093
Итерация 3: K=3, точность=0,7897
Итерация 4: K=15, точность=0,8164
Итерация 5: K=5, точность=0,8093
Итерация 6: K=7, точность=0,8178
Итерация 7: K=21, точность=0,8178
Итерация 8: K=15, точность=0,8164
Итерация 9: K=5, точность=0,8093
Итерация 10: K=23, точность=0,8178

Лучший K (Randomized): 23 с точностью 0,8178


<!-- Plotly chart will be drawn inside this DIV -->

## 6. Сравнение качества моделей

In [18]:
// Модель с лучшим K из GridSearch
var knnGrid = new KNNClassifier(bestK_Grid);
knnGrid.Fit(X_train, y_train);
var predGrid = knnGrid.Predict(X_test);
double accGrid = Accuracy(y_test, predGrid);

// Модель с лучшим K из RandomizedSearch
var knnRand = new KNNClassifier(bestK_Rand);
knnRand.Fit(X_train, y_train);
var predRand = knnRand.Predict(X_test);
double accRand = Accuracy(y_test, predRand);

Console.WriteLine("=== Итоговое сравнение ===");
Console.WriteLine($"Базовая модель (K=5)          : точность = {baseAcc:F4}");
Console.WriteLine($"GridSearchCV     (K={bestK_Grid}) : точность = {accGrid:F4}");
Console.WriteLine($"RandomizedSearch (K={bestK_Rand}) : точность = {accRand:F4}");

// Сравнительная столбчатая диаграмма
var models = new[] { "K=5 (базовая)", $"K={bestK_Grid} (Grid)", $"K={bestK_Rand} (Random)" };
var accuracies = new[] { baseAcc, accGrid, accRand };
var barFinal = PlotlyCS.Column<string, double, string>(models, accuracies)
    .WithTitle("Сравнение точности моделей")
    .WithYAxisStyle(Title.init("Accuracy"));
display(barFinal);

=== Итоговое сравнение ===
Базовая модель (K=5)          : точность = 0,7809
GridSearchCV     (K=11) : точность = 0,7865
RandomizedSearch (K=23) : точность = 0,7697


<!-- Plotly chart will be drawn inside this DIV -->

## 7. Заключение

В ходе работы:

- Подготовлен датасет Titanic: заполнены пропуски, закодированы категориальные признаки, выполнено масштабирование.
- Реализован алгоритм k‑ближайших соседей с нуля.
- Данные разделены на обучающую (80%) и тестовую (20%) выборки.
- Проведён подбор гиперпараметра *K* двумя методами:
  - **GridSearchCV** (полный перебор нечётных K от 1 до 29 с 5‑фолдовой кросс-валидацией);
  - **RandomizedSearchCV** (10 случайных попыток из того же диапазона).
- Лучшие модели показали более высокую точность на тестовой выборке по сравнению с базовой (K=5).

Таким образом, подбор гиперпараметров с кросс-валидацией помогает повысить качество классификации для метода k‑ближайших соседей.